# Freemarket Medallion Pipeline — orchestrator

Thin orchestrator: all logic lives in the tested `src/` package (see `plan/ARCHITECTURE.md`).
Run top-to-bottom (Restart & Run All) from an empty warehouse.

Bronze `live` holds the facts + counterparty; Silver `core` = dims + FX match; Silver `shape`
= resolved attributes + GBP-normalised facts; Gold `data_mart` = the modelled network.

In [1]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import config, warehouse, bronze, silver_core, silver_shape, gold
from src.pipeline import build_foundation
config.WAREHOUSE_PATH

PosixPath('/Users/hallabehery/Documents/My Projects/FM-Data-Engineer-Test-HS/submission/warehouse.duckdb')

## 1. Foundation — six schemas

In [2]:
con = warehouse.connect()
build_foundation(con)

21:58:50 | INFO  | [foundation] schemas ready: raw, live, core, shape, data_mart, curated


('raw', 'live', 'core', 'shape', 'data_mart', 'curated')

## 2. Bronze `raw` — transactional sheets split per month

In [3]:
for r in bronze.build_raw_monthly(con):
    print(r)

21:58:50 | INFO  | [bronze.raw.deposit] in=6383 out=6383 kept=6383 quarantined=0 dropped=0 conserved=yes


21:58:50 | INFO  | [bronze.raw.withdrawal] in=6599 out=6599 kept=6599 quarantined=0 dropped=0 conserved=yes


[bronze.raw.deposit] in=6383 out=6383 kept=6383 quarantined=0 dropped=0 conserved=yes
[bronze.raw.withdrawal] in=6599 out=6599 kept=6599 quarantined=0 dropped=0 conserved=yes


In [4]:
con.sql("SELECT * FROM raw.deposit_2025_07 LIMIT 10").df()

,freemarket_entity,transaction_type,dc_id,account_id,transaction_id,tx_date,tx_time,tx_currency,tx_value_ccy,counterparty_id,tx_month,scheme
0,UK,Deposit,234939159769,16472.0,343316.0,2025-07-30,13:38:31.593000,USD,2.438954e+06,CP6,2025-07-01,Swift
1,UK,Deposit,234939159769,16472.0,341662.0,2025-07-24,10:39:47.117000,USD,2.736424e+05,CP5,2025-07-01,Swift
2,UK,Deposit,234939159769,16472.0,338562.0,2025-07-14,12:16:16.897000,USD,9.515370e+04,CP5,2025-07-01,Swift
3,UK,Deposit,234939159769,16472.0,336583.0,2025-07-07,12:52:57.710000,USD,1.627457e+06,CP6,2025-07-01,Swift
4,UK,Deposit,234939159769,16472.0,340011.0,2025-07-18,12:50:36.097000,USD,1.209455e+06,CP5,2025-07-01,Swift
5,UK,Deposit,234939159769,16472.0,338879.0,2025-07-15,10:47:03.707000,USD,8.258716e+05,CP6,2025-07-01,Swift
6,UK,Deposit,263002682585,16968.0,339726.0,2025-07-17,15:33:15.537001,EUR,6.032883e+04,CP12,2025-07-01,Sepa
7,UK,Deposit,234850008297,16627.0,340931.0,2025-07-22,09:36:20.080000,USD,4.591582e+04,CP18,2025-07-01,Swift
8,UK,Deposit,263011742938,16914.0,343561.0,2025-07-31,10:18:41.933000,EUR,2.924164e+05,CP39,2025-07-01,SepaInstant
9,UK,Deposit,263011742938,16914.0,337205.0,2025-07-09,09:09:09.650000,EUR,3.899708e+05,CP39,2025-07-01,SepaInstant


## 3. Bronze `live` — consolidate + land counterparty & fee (facts live here)

In [5]:
for r in bronze.build_live(con):
    print(r)

21:58:50 | INFO  | [bronze.live.deposit] in=6383 out=6383 kept=6383 quarantined=0 dropped=0 conserved=yes


21:58:50 | INFO  | [bronze.live.withdrawal] in=6599 out=6599 kept=6599 quarantined=0 dropped=0 conserved=yes


21:58:50 | INFO  | [bronze.live.counterparty] in=1585 out=1585 kept=1585 quarantined=0 dropped=0 conserved=yes


21:58:51 | INFO  | [bronze.live.fee] in=21921 out=21921 kept=21921 quarantined=0 dropped=0 conserved=yes


[bronze.live.deposit] in=6383 out=6383 kept=6383 quarantined=0 dropped=0 conserved=yes
[bronze.live.withdrawal] in=6599 out=6599 kept=6599 quarantined=0 dropped=0 conserved=yes
[bronze.live.counterparty] in=1585 out=1585 kept=1585 quarantined=0 dropped=0 conserved=yes
[bronze.live.fee] in=21921 out=21921 kept=21921 quarantined=0 dropped=0 conserved=yes


In [6]:
con.sql("SELECT table_name FROM duckdb_tables() WHERE schema_name = 'live' ORDER BY table_name").df()

,table_name
0,counterparty
1,deposit
2,fee
3,withdrawal


In [7]:
con.sql("SELECT * FROM live.deposit LIMIT 10").df()

,freemarket_entity,transaction_type,dc_id,account_id,transaction_id,tx_date,tx_time,tx_currency,tx_value_ccy,counterparty_id,tx_month,scheme
0,UK,Deposit,234939159769,16472.0,343316.0,2025-07-30,13:38:31.593000,USD,2.438954e+06,CP6,2025-07-01,Swift
1,UK,Deposit,234939159769,16472.0,341662.0,2025-07-24,10:39:47.117000,USD,2.736424e+05,CP5,2025-07-01,Swift
2,UK,Deposit,234939159769,16472.0,338562.0,2025-07-14,12:16:16.897000,USD,9.515370e+04,CP5,2025-07-01,Swift
3,UK,Deposit,234939159769,16472.0,336583.0,2025-07-07,12:52:57.710000,USD,1.627457e+06,CP6,2025-07-01,Swift
4,UK,Deposit,234939159769,16472.0,340011.0,2025-07-18,12:50:36.097000,USD,1.209455e+06,CP5,2025-07-01,Swift
5,UK,Deposit,234939159769,16472.0,338879.0,2025-07-15,10:47:03.707000,USD,8.258716e+05,CP6,2025-07-01,Swift
6,UK,Deposit,263002682585,16968.0,339726.0,2025-07-17,15:33:15.537001,EUR,6.032883e+04,CP12,2025-07-01,Sepa
7,UK,Deposit,234850008297,16627.0,340931.0,2025-07-22,09:36:20.080000,USD,4.591582e+04,CP18,2025-07-01,Swift
8,UK,Deposit,263011742938,16914.0,343561.0,2025-07-31,10:18:41.933000,EUR,2.924164e+05,CP39,2025-07-01,SepaInstant
9,UK,Deposit,263011742938,16914.0,337205.0,2025-07-09,09:09:09.650000,EUR,3.899708e+05,CP39,2025-07-01,SepaInstant


## 4. Silver `core` — company & corporate_group unpick (dimensions)

In [8]:
print(silver_core.build_companies(con))
print(silver_core.build_groups(con))

21:58:51 | INFO  | [silver.core.company] in=44 out=44 kept=44 quarantined=0 dropped=0 conserved=yes


21:58:51 | INFO  | [silver.core.corporate_group] in=13 out=13 kept=13 quarantined=0 dropped=0 conserved=yes


[silver.core.company] in=44 out=44 kept=44 quarantined=0 dropped=0 conserved=yes
[silver.core.corporate_group] in=13 out=13 kept=13 quarantined=0 dropped=0 conserved=yes


In [9]:
con.sql("SELECT * FROM core.company LIMIT 10").df()

,dc_id,legal_name,status,incorporation_country,incorporation_iso2,incorporated_on,created_epoch_ms,vertical,industry,licensed,headcount,headcount_basis,financials_currency,annual_revenue,total_assets,operation_count,countries_of_operation,parent_group_id,parent_group_role,attributes_json
0,234850008297,Payment One Holding S.à r.l.,ACTIVE,Cyprus,CY,2024-04-26,1714089600000,Holding,Holding Company / Group Treasury,False,25,FTE,USD,USD 5.95M,USD 714.88M,6,"[BR, SE, AE, AU, IN, NL]",167898894567,HOLDING,"[{""name"":""risk_level"",""value"":""Medium""},{""name..."
1,234939159769,Payment One Operating Company Ltd,ACTIVE,Cyprus,CY,2024-01-18,1705536000000,MSB,Money Service Business,True,490,FTE,USD,USD 91.70M,USD 187.04M,6,"[PL, DE, IN, SE, PT, US]",167898894567,OPERATING,"[{""name"":""risk_level"",""value"":""High""},{""name"":..."
2,234854669542,Remitance Central Holdings Ltd,SUSPENDED,Lithuania,LT,2022-12-08,1670457600000,Holding,Holding Company / Group Treasury,False,30,FTE,USD,USD 10.24M,USD 321.26M,4,"[AE, CA, US, SG]",167986897142,HOLDING,"[{""name"":""risk_level"",""value"":""Low""},{""name"":""..."
3,234939159775,Remitance Central (Europe) Ltd,ACTIVE,Curaçao,CW,2021-10-20,1634688000000,MSB,Money Service Business,True,600,FTE,USD,USD 303.60M,USD 82.26M,2,"[JP, GB]",167986897142,OPERATING,"[{""name"":""risk_level"",""value"":""Low""},{""name"":""..."
4,258599414982,Get Tranzzact (Europe) Ltd,ACTIVE,Luxembourg,LU,2023-08-31,1693440000000,MSB,Money Service Business,True,395,FTE,USD,USD 171.08M,USD 187.81M,1,[SG],187752508616,OPERATING,"[{""name"":""risk_level"",""value"":""Low""},{""name"":""..."
5,258991877340,ConstructGame Operating Company Ltd,ACTIVE,Netherlands,NL,2021-06-21,1624233600000,Gaming,Online Gaming & Gambling,True,360,FTE,USD,USD 245.07M,USD 248.84M,2,"[DE, MX]",187754310858,OPERATING,"[{""name"":""risk_level"",""value"":""Low""},{""name"":""..."
6,258970163419,ConstructGame Payments UAB,ACTIVE,Gibraltar,GI,2023-04-17,1681689600000,Payments,Payment & E-Money Services,True,200,FTE,USD,USD 129.64M,USD 161.72M,6,"[AE, ZA, GB, IE, MX, SG]",187754310858,PAYMENTS,"[{""name"":""risk_level"",""value"":""High""},{""name"":..."
7,258740050135,ConstructGame Technology Pvt Ltd,ACTIVE,Curaçao,CW,2024-11-13,1731456000000,Technology,Software Development & Platform Engineering,False,365,FTE,USD,USD 42.56M,USD 29.54M,6,"[ZA, PL, NL, IE, GB, DE]",187754310858,TECHNOLOGY,"[{""name"":""risk_level"",""value"":""Low""},{""name"":""..."
8,258929414331,ConstructGame Media Ltd,SUSPENDED,Gibraltar,GI,2024-04-19,1713484800000,Marketing,Marketing & Customer Acquisition,False,60,FTE,USD,USD 26.44M,USD 5.46M,4,"[US, CA, IE, SG]",187754310858,MARKETING,"[{""name"":""risk_level"",""value"":""Medium""},{""name..."
9,279464884442,ConstructGame IP Ltd,SUSPENDED,Estonia,EE,2025-08-19,1755561600000,Licensing,Intellectual Property Licensing,False,5,FTE,USD,USD 42.87M,USD 218.77M,5,"[MX, PL, JP, AE, AU]",187754310858,LICENSING_IP,"[{""name"":""risk_level"",""value"":""Low""},{""name"":""..."


## 5. Silver `core` — `exchange_rate` dimension

In [10]:
print(silver_core.build_exchange_rates(con))

21:58:51 | INFO  | [silver.core.exchange_rate] in=161342 out=161342 kept=161342 quarantined=0 dropped=0 conserved=yes


[silver.core.exchange_rate] in=161342 out=161342 kept=161342 quarantined=0 dropped=0 conserved=yes


## 6. Silver `core` — FX as-of match (facts stay in `live`)

In [11]:
for r in silver_core.build_fx_match(con):
    print(r)

21:58:52 | INFO  | [silver.core.deposit_fx] in=6383 out=6383 kept=6383 quarantined=0 dropped=0 conserved=yes


21:58:52 | INFO  | [silver.core.withdrawal_fx] in=6599 out=6599 kept=6599 quarantined=0 dropped=0 conserved=yes


21:58:52 | INFO  | [silver.core.fee_fx] in=21921 out=21921 kept=21921 quarantined=0 dropped=0 conserved=yes


21:58:52 | INFO  | [silver.core.fee_fx] flagged for FX quarantine (kept, not dropped): 42


[silver.core.deposit_fx] in=6383 out=6383 kept=6383 quarantined=0 dropped=0 conserved=yes
[silver.core.withdrawal_fx] in=6599 out=6599 kept=6599 quarantined=0 dropped=0 conserved=yes
[silver.core.fee_fx] in=21921 out=21921 kept=21921 quarantined=0 dropped=0 conserved=yes


In [12]:
con.sql("SELECT table_name FROM duckdb_tables() WHERE schema_name = 'core' ORDER BY table_name").df()

,table_name
0,company
1,corporate_group
2,deposit_fx
3,exchange_rate
4,fee_fx
5,withdrawal_fx


In [13]:
con.sql("SELECT * FROM core.deposit_fx LIMIT 10").df()

,transaction_id,fx_instant_ms,fx_rate_id,fx_rate,fx_quarantine_reason
0,343316.0,1753882711593,13848698,0.752064,<NA>
1,341662.0,1753353587117,13832598,0.738758,<NA>
2,338562.0,1752495376897,13801564,0.741411,<NA>
3,336583.0,1751892777710,13781902,0.734611,<NA>
4,340011.0,1752843036097,13817325,0.743028,<NA>
5,338879.0,1752576423707,13805164,0.743192,<NA>
6,339726.0,1752766395537,13813856,0.864524,<NA>
7,340931.0,1753176980080,13824631,0.741222,<NA>
8,343561.0,1753957121933,13851592,0.864840,<NA>
9,337205.0,1752052149650,13788912,0.861393,<NA>


## 7. Silver `shape` — resolve heterogeneous entity attributes

In [14]:
for r in silver_shape.build_entity_shape(con):
    print(r)

21:58:52 | INFO  | [silver.shape.company] in=44 out=44 kept=44 quarantined=0 dropped=0 conserved=yes


21:58:52 | INFO  | [silver.shape.corporate_group] in=13 out=13 kept=13 quarantined=0 dropped=0 conserved=yes


[silver.shape.company] in=44 out=44 kept=44 quarantined=0 dropped=0 conserved=yes
[silver.shape.corporate_group] in=13 out=13 kept=13 quarantined=0 dropped=0 conserved=yes


In [15]:
con.sql("SELECT * FROM shape.corporate_group LIMIT 10").df()

,group_id,display_name,description,status_code,active,created_on,pod,vertical,industry,commercial_tier,attr_risk_level,attr_country_of_incorporation_label,attr_country_of_incorporation_iso2
0,167898894567,Payment One,Payment One is a Gold-tier money service busin...,ACTIVE,True,2024-01-18,MSB/PSP,MSB,Money Service Business,Gold,Low,Ireland,IE
1,167986897142,Remitance Central,Remitance Central is a Bronze-tier money servi...,ACTIVE,True,2021-10-20,MSB/PSP,MSB,Money Service Business,Bronze,Low,United Kingdom,GB
2,187752508616,Get Tranzzact,Get Tranzzact is a Gold-tier money service bus...,SUSPENDED,False,2023-08-31,MSB/PSP,MSB,Money Service Business,Gold,High,Luxembourg,LU
3,187754310858,ConstructGame,ConstructGame is a Gold-tier online gaming & g...,ACTIVE,True,2021-06-21,Gaming/iGaming/Gambling,Gaming,Online Gaming & Gambling,Gold,Medium,Cyprus,CY
4,187757905092,Sanno Payments,Sanno Payments is a Silver-tier money service ...,ACTIVE,True,2024-06-28,MSB/PSP,MSB,Money Service Business,Silver,Medium,Lithuania,LT
5,192212183238,Tech Key,Tech Key is a Bronze-tier money service busine...,ACTIVE,True,2025-02-27,MSB/PSP,MSB,Money Service Business,Bronze,High,Lithuania,LT
6,192230405343,Koba2Maya,Koba2Maya is a Silver-tier online gaming & gam...,ACTIVE,True,2020-06-22,Gaming/iGaming/Gambling,Gaming,Online Gaming & Gambling,Silver,Low,Curaçao,CW
7,192315275482,Oni Group,Oni Group is a Bronze-tier money service busin...,ACTIVE,True,2023-12-13,MSB/PSP,MSB,Money Service Business,Bronze,Low,United Kingdom,GB
8,192400133342,NXT Services,NXT Services is a Silver-tier crypto assets & ...,ACTIVE,True,2024-09-30,Crypto/CFD/Forex,Crypto,Crypto Assets & Leveraged Trading,Silver,Medium,Cyprus,CY
9,192402023626,Cube Empire,Cube Empire is a Bronze-tier money service bus...,SUSPENDED,False,2024-12-03,MSB/PSP,MSB,Money Service Business,Bronze,Low,United Kingdom,GB


## 8. Silver `shape` — FX applied → GBP normalisation

In [16]:
for r in silver_shape.build_gbp_facts(con):
    print(r)

21:58:52 | INFO  | [silver.shape.deposit] in=6383 out=6383 kept=6383 quarantined=0 dropped=0 conserved=yes


21:58:52 | INFO  | [silver.shape.withdrawal] in=6599 out=6599 kept=6599 quarantined=0 dropped=0 conserved=yes


21:58:52 | INFO  | [silver.shape.fee] in=21921 out=21879 kept=21879 quarantined=42 dropped=0 conserved=yes


[silver.shape.deposit] in=6383 out=6383 kept=6383 quarantined=0 dropped=0 conserved=yes
[silver.shape.withdrawal] in=6599 out=6599 kept=6599 quarantined=0 dropped=0 conserved=yes
[silver.shape.fee] in=21921 out=21879 kept=21879 quarantined=42 dropped=0 conserved=yes


In [17]:
con.sql("SELECT table_name FROM duckdb_tables() WHERE schema_name = 'shape' ORDER BY table_name").df()

,table_name
0,company
1,corporate_group
2,deposit
3,deposit_quarantine
4,fee
5,fee_quarantine
6,withdrawal
7,withdrawal_quarantine


In [18]:
con.sql("SELECT * FROM shape.deposit LIMIT 10").df()

,freemarket_entity,transaction_type,dc_id,account_id,transaction_id,tx_date,tx_time,tx_currency,tx_value_ccy,counterparty_id,tx_month,scheme,fx_rate_id,fx_rate,gbp_amount
0,UK,Deposit,234939159769,16472.0,343316.0,2025-07-30,13:38:31.593000,USD,2.438954e+06,CP6,2025-07-01,Swift,13848698,0.752064,1.834249e+06
1,UK,Deposit,234939159769,16472.0,341662.0,2025-07-24,10:39:47.117000,USD,2.736424e+05,CP5,2025-07-01,Swift,13832598,0.738758,2.021556e+05
2,UK,Deposit,234939159769,16472.0,338562.0,2025-07-14,12:16:16.897000,USD,9.515370e+04,CP5,2025-07-01,Swift,13801564,0.741411,7.054805e+04
3,UK,Deposit,234939159769,16472.0,336583.0,2025-07-07,12:52:57.710000,USD,1.627457e+06,CP6,2025-07-01,Swift,13781902,0.734611,1.195547e+06
4,UK,Deposit,234939159769,16472.0,340011.0,2025-07-18,12:50:36.097000,USD,1.209455e+06,CP5,2025-07-01,Swift,13817325,0.743028,8.986596e+05
5,UK,Deposit,234939159769,16472.0,338879.0,2025-07-15,10:47:03.707000,USD,8.258716e+05,CP6,2025-07-01,Swift,13805164,0.743192,6.137808e+05
6,UK,Deposit,263002682585,16968.0,339726.0,2025-07-17,15:33:15.537001,EUR,6.032883e+04,CP12,2025-07-01,Sepa,13813856,0.864524,5.215570e+04
7,UK,Deposit,234850008297,16627.0,340931.0,2025-07-22,09:36:20.080000,USD,4.591582e+04,CP18,2025-07-01,Swift,13824631,0.741222,3.403382e+04
8,UK,Deposit,263011742938,16914.0,343561.0,2025-07-31,10:18:41.933000,EUR,2.924164e+05,CP39,2025-07-01,SepaInstant,13851592,0.864840,2.528935e+05
9,UK,Deposit,263011742938,16914.0,337205.0,2025-07-09,09:09:09.650000,EUR,3.899708e+05,CP39,2025-07-01,SepaInstant,13788912,0.861393,3.359181e+05


## 9. Gold `data_mart` — combined entity/node dimension
Groups + companies (Silver) combined with counterparties (`live`). Counterparties resolve to
a group via `group_id` or `dc_id → company → parent group`, else stand alone. `node_shape` is
`circle` (group) / `diamond` (standalone counterparty) / NULL (drill or resolved). Each row
carries a `source` provenance list.

In [19]:
print(gold.build_entity(con))

21:58:52 | INFO  | [gold.data_mart.entity] in=1642 out=1642 kept=1642 quarantined=0 dropped=0 conserved=yes


[gold.data_mart.entity] in=1642 out=1642 kept=1642 quarantined=0 dropped=0 conserved=yes


In [20]:
con.sql("SELECT table_name FROM duckdb_tables() WHERE schema_name = 'data_mart' ORDER BY table_name").df()

,table_name
0,entity


In [21]:
# entity mix: kind × node_shape
con.sql('''SELECT entity_kind, node_shape, is_standalone, COUNT(*) n
           FROM data_mart.entity GROUP BY 1,2,3 ORDER BY 1,2''').df()

,entity_kind,node_shape,is_standalone,n
0,company,NaN,False,44
1,counterparty,diamond,True,1363
2,counterparty,NaN,False,222
3,group,circle,False,13


In [22]:
con.sql("SELECT * FROM data_mart.entity LIMIT 10").df()

,entity_id,entity_kind,name,group_id,is_standalone,node_shape,source
0,167898894567,group,Payment One,167898894567,False,circle,[groups.json]
1,167986897142,group,Remitance Central,167986897142,False,circle,[groups.json]
2,187752508616,group,Get Tranzzact,187752508616,False,circle,[groups.json]
3,187754310858,group,ConstructGame,187754310858,False,circle,[groups.json]
4,187757905092,group,Sanno Payments,187757905092,False,circle,[groups.json]
5,192212183238,group,Tech Key,192212183238,False,circle,[groups.json]
6,192230405343,group,Koba2Maya,192230405343,False,circle,[groups.json]
7,192315275482,group,Oni Group,192315275482,False,circle,[groups.json]
8,192400133342,group,NXT Services,192400133342,False,circle,[groups.json]
9,192402023626,group,Cube Empire,192402023626,False,circle,[groups.json]


## Inspect the built warehouse — table counts per schema

In [23]:
con.close()
import duckdb
inspect = duckdb.connect(str(config.WAREHOUSE_PATH), read_only=True)
tables = inspect.sql('''SELECT table_schema, COUNT(*) AS n_tables
    FROM information_schema.tables GROUP BY table_schema ORDER BY table_schema''').df()
inspect.close()
tables

,table_schema,n_tables
0,core,6
1,data_mart,1
2,live,4
3,raw,12
4,shape,8
